# Importing Pandas and loading the data

In [1]:
import pandas as pd
import numpy as np
import re

orders_df = pd.read_csv('../data/raw/orders.csv')
orderlines_df = pd.read_csv('../data/raw/orderlines.csv')
products_df = pd.read_csv('../data/raw/products.csv')
brands_df = pd.read_csv('../data/raw/brands.csv')

In [2]:
def clean_unit_prices(x):
    #Clean malformed unit_price values by keeping only the last decimal separator
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # Keep only numbers, '.' and ','
    s = re.sub(r"[^0-9\.,]", "", s)

    # Convert ',' to '.'
    s = s.replace(",", ".")

    # Keep only the last '.' as decimal separator
    if s.count(".") > 1:
        parts = s.split(".")
        s = "".join(parts[:-1]) + "." + parts[-1]
    
    try:
        val = float(s)
    except ValueError:
        return np.nan

    return str(val)

In [3]:
def clean_prices_v2(x):
    #Clean malformed product price values and correct values with three decimal digits
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    # Keep only numbers, '.' and ','
    s = re.sub(r"[^0-9\.,]", "", s)

    # Convert ',' to '.'
    s = s.replace(",", ".")

    # Keep only the last '.' as decimal separator
    if s.count(".") > 1:
        parts = s.split(".")
        s = "".join(parts[:-1]) + "." + parts[-1]

    try:
        val = float(s)
    except ValueError:
        return np.nan

    # If there are exactly three digits after '.', the value is likely scaled by 10
    if "." in s:
        dec = s.split(".")[-1]
        if len(dec) == 3:
            val = val / 10

    return str(val)

In [4]:
def clean_still_bad_promo_prices(x):
    #Correct remaining product-level prices that still appear to be scaled by 10
    return x / 10

## Making copies

In [5]:
or_df = orders_df.copy()
ol_df = orderlines_df.copy()
pr_df = products_df.copy()
br_df = brands_df.copy()

## Cleaning Orders DataFrame :

We are interested only in Completed Orders

In [6]:
or_df.shape

(226909, 4)

In [7]:
or_df['state'].value_counts(normalize=True)

state
Shopping Basket    0.519191
Completed          0.205391
Place Order        0.180174
Pending            0.063369
Cancelled          0.031876
Name: proportion, dtype: float64

In [8]:
or_df = or_df.loc[or_df['state'] == 'Completed']

In [9]:
or_df.info()

<class 'pandas.DataFrame'>
Index: 46605 entries, 1 to 226619
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      46605 non-null  int64  
 1   created_date  46605 non-null  str    
 2   total_paid    46605 non-null  float64
 3   state         46605 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 1.8 MB


In [10]:
# now we need to delete orders that do not appear in order lines, although they are completed orders
or_df.loc[~or_df['order_id'].isin(ol_df['id_order'])]

,order_id,created_date,total_paid,state
8,245941,2017-01-01 10:32:23,183.52,Completed
65,268629,2017-01-31 11:27:25,73.23,Completed
83,277994,2017-01-23 18:30:11,52.99,Completed
96,282180,2018-01-06 16:28:49,59.99,Completed
100,284200,2017-01-17 13:43:48,49.94,Completed
114,287780,2017-01-01 21:32:41,94.23,Completed
123,290018,2017-01-05 15:07:06,819.99,Completed
124,290223,2017-01-12 12:16:07,171.98,Completed
128,290487,2017-01-08 23:04:13,19.97,Completed
132,291378,2017-01-06 12:15:18,34.97,Completed


In [11]:
or_df = or_df.loc[or_df['order_id'].isin(ol_df['id_order'])]
or_df.info()

<class 'pandas.DataFrame'>
Index: 46560 entries, 1 to 226619
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      46560 non-null  int64  
 1   created_date  46560 non-null  str    
 2   total_paid    46560 non-null  float64
 3   state         46560 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 1.8 MB


In [12]:
# Deleting order lines that do not belong to completed orders.
# Now that we only focus on completed orders, we keep only their order lines.
ol_df = ol_df.loc[ol_df['id_order'].isin(or_df['order_id'])].copy()
ol_df.info()

<class 'pandas.DataFrame'>
Index: 62103 entries, 7 to 293661
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id                62103 non-null  int64
 1   id_order          62103 non-null  int64
 2   product_id        62103 non-null  int64
 3   product_quantity  62103 non-null  int64
 4   sku               62103 non-null  str  
 5   unit_price        62103 non-null  str  
 6   date              62103 non-null  str  
dtypes: int64(4), str(3)
memory usage: 3.8 MB


In [13]:
# converting created_date to datetime in orders DF
or_df['created_date'] = pd.to_datetime(or_df['created_date'])

In [14]:
or_df.info()

<class 'pandas.DataFrame'>
Index: 46560 entries, 1 to 226619
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      46560 non-null  int64         
 1   created_date  46560 non-null  datetime64[us]
 2   total_paid    46560 non-null  float64       
 3   state         46560 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 1.8 MB


In [15]:
# check for duplicates
or_df.duplicated().sum()

np.int64(0)

## Cleaning Products DataFrame :

### Eliminating duplicates :

In [16]:
pr_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          19326 non-null  str  
 1   name         19326 non-null  str  
 2   desc         19319 non-null  str  
 3   price        19280 non-null  str  
 4   promo_price  19326 non-null  str  
 5   in_stock     19326 non-null  int64
 6   type         19276 non-null  str  
dtypes: int64(1), str(6)
memory usage: 1.0 MB


In [17]:
# checking for duplicates :
pr_df.duplicated('sku').sum()

np.int64(8747)

In [18]:
# From the previous cell, we can see that we have duplicated SKUs and need to drop them.
pr_df = pr_df.drop_duplicates(subset=['sku'], ignore_index=True)
pr_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10579 entries, 0 to 10578
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          10579 non-null  str  
 1   name         10579 non-null  str  
 2   desc         10572 non-null  str  
 3   price        10534 non-null  str  
 4   promo_price  10579 non-null  str  
 5   in_stock     10579 non-null  int64
 6   type         10529 non-null  str  
dtypes: int64(1), str(6)
memory usage: 578.7 KB


In [19]:
pr_df['sku'].nunique()

10579

In [20]:
# in Orderlines DF, we have a column (product_id), which does not exist in products DataFrame !!
ol_df['product_id'].value_counts()
# product_id contains only zeros and therefore carries no analytical value
ol_df = ol_df.drop('product_id', axis=1)

In [21]:
# converting date to datetime in order lines df
ol_df['date'] = pd.to_datetime(ol_df['date'])

### Dropping never sold products :

In [22]:
# so that we are interested in sold/(in completed orders) products
# let us drop never sold products
never_sold_products = pr_df.loc[~pr_df['sku'].isin(ol_df['sku'])]
never_sold_products

,sku,name,desc,price,promo_price,in_stock,type
6,KIN0008,Mac Memory Kingston 1GB 667MHz DDR2 SO-DIMM,1GB RAM Mac mini and iMac (2006/07) MacBook Pr...,18.99,146.471,0,1364
7,KIN0009,Mac Memory Kingston 2GB 800MHz DDR2 SO-DIMM,2GB RAM iMac with Intel Core 2 Duo (Penryn).,36.99,274.694,0,1364
11,SEN0021,Sennheiser CX 300-II Precision headphones iPho...,Headphones iPhone iPad iPad 2 iPad 3 and iPod.,49.99,449.878,0,5384
18,MUV0009,Muvit Back Clear Case iPhone 4 / 4S Transparent,Case iPhone 4 / 4S polycarbonate.,9.99,7.986,0,11865403
21,APP0233,Apple iPad Camera Connection connector,IPad connector for digital cameras and SD cards.,35,349.896,0,13955395
...,...,...,...,...,...,...,...
10574,BEL0376,Belkin Travel Support Apple Watch Black,compact and portable stand vertically or horiz...,29.99,269.903,1,12282
10575,THU0060,"Enroute Thule 14L Backpack MacBook 13 ""Black",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
10576,THU0061,"Enroute Thule 14L Backpack MacBook 13 ""Blue",Backpack with capacity of 14 liter compartment...,69.95,649.903,1,1392
10577,THU0062,"Enroute Thule 14L Backpack MacBook 13 ""Red",Backpack with capacity of 14 liter compartment...,69.95,649.903,0,1392


In [23]:
pr_df = pr_df.loc[~pr_df['sku'].isin(never_sold_products['sku'])]
pr_df.info()

<class 'pandas.DataFrame'>
Index: 5985 entries, 0 to 10535
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          5985 non-null   str  
 1   name         5985 non-null   str  
 2   desc         5981 non-null   str  
 3   price        5970 non-null   str  
 4   promo_price  5985 non-null   str  
 5   in_stock     5985 non-null   int64
 6   type         5976 non-null   str  
dtypes: int64(1), str(6)
memory usage: 374.1 KB


### Working with Null values : 

In [24]:
# checking null values
pr_df.isnull().sum()

sku             0
name            0
desc            4
price          15
promo_price     0
in_stock        0
type            9
dtype: int64

In [25]:
# Null values in desc and type columns
pr_df['desc'] = pr_df['desc'].fillna(pr_df['name'])
pr_df['type'] = pr_df['type'].fillna('unknown')
pr_df.info()

<class 'pandas.DataFrame'>
Index: 5985 entries, 0 to 10535
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          5985 non-null   str  
 1   name         5985 non-null   str  
 2   desc         5985 non-null   str  
 3   price        5970 non-null   str  
 4   promo_price  5985 non-null   str  
 5   in_stock     5985 non-null   int64
 6   type         5985 non-null   str  
dtypes: int64(1), str(6)
memory usage: 374.1 KB


In [26]:
# null values in price column, we can fill them with values from promo_price
# price_is_null_mask = pr_df['price'].isna()
pr_df.loc[pr_df['price'].isna()]

,sku,name,desc,price,promo_price,in_stock,type
1720,CEL0007,Celly Wallet Case with removable cover Black i...,Case Book for iPhone 6 card case type.,NaN,128.998,0,11865403
1721,CEL0012,Celly Silicone Hard Shell iPhone 6 Blue,Hard Shell Silicone iPhone 6.,NaN,4.99,0,11865403
1730,CEL0023,Celly Ambo Luxury Leather Case + Case Gold iPh...,Cover and housing together with magnet for iPh...,NaN,329.894,0,11865403
4292,CEL0028,Celly Swing Arm button selfie with Bluetooth B...,selfie extendable arm for iPhone,NaN,59.895,0,11905404
4293,CEL0029,Celly extensible arm selfie with Bluetooth but...,selfie extendable arm for iPhone,NaN,69.902,0,5720
4301,CEL0043,Celly Fluor Case iPhone 6 / 6S Pink / Transparent,Protective Case for iPhone 6 and 6s,NaN,99.898,0,11865403
4302,CEL0044,Celly iPhone Case Fluor 6s / 6 Yellow / Transp...,protective case with two colors for iPhone 6 a...,NaN,79.896,1,11865403
4307,CEL0033,"Celly Waterproof iPad Case / Air / Pro 97 ""Tra...",waterproof protective case for iPad bag format...,NaN,149.895,0,12635403
4310,CEL0036,Celly Adapter Micro USB connector to Lightning...,Micro USB adapter cable connection Lightning,NaN,89.903,1,14365395
4312,CEL0052,Celly 4000mAh Battery Power Bank Aluminum Silver,external battery capacity 4000mAh output volta...,NaN,199.892,0,1515


In [27]:
# The promo_price values in the previous rows look usable, so we use them to fill missing prices.
pr_df.loc[pr_df['price'].isna(), 'price'] = pr_df.loc[pr_df['price'].isna(), 'promo_price']

In [28]:
pr_df.loc[pr_df['price'].isna()]

,sku,name,desc,price,promo_price,in_stock,type


In [29]:
pr_df.info()

<class 'pandas.DataFrame'>
Index: 5985 entries, 0 to 10535
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          5985 non-null   str  
 1   name         5985 non-null   str  
 2   desc         5985 non-null   str  
 3   price        5985 non-null   str  
 4   promo_price  5985 non-null   str  
 5   in_stock     5985 non-null   int64
 6   type         5985 non-null   str  
dtypes: int64(1), str(6)
memory usage: 374.1 KB


In [30]:
# let's check if all products in order_lines DF appear in Products DF
temp = ol_df.loc[~ol_df['sku'].isin(pr_df['sku'])]
temp

,id,id_order,product_quantity,sku,unit_price,date
117,1119316,299638,1,SYN0127,223.24,2017-01-01 11:56:18
192,1119477,299706,1,EVU0007,28.49,2017-01-01 13:57:16
198,1119494,299712,1,APP0608,279.99,2017-01-01 14:10:47
510,1120084,300029,1,SYN0127,223.24,2017-01-01 22:00:19
752,1120534,300251,1,APP0608,279.99,2017-01-02 07:48:49
...,...,...,...,...,...,...
240284,1564525,492329,1,AP20172,559.00,2018-01-16 19:26:20
241823,1567517,493528,3,WDT0177-A,123.03,2018-01-18 12:40:42
263415,1601131,491926,1,REP0088,479.65,2018-02-05 13:17:51
263416,1601133,491926,1,REP0088,14.31,2018-02-05 13:20:30


In [31]:
pr_df.loc[pr_df['sku'].isin(temp['sku'])]

,sku,name,desc,price,promo_price,in_stock,type


In [32]:
# this means we need to drop the whole order not just one line from the order 
temp['id_order'].nunique()

356

In [33]:
# now we need to delete these 356 orders(the whole order) from order lines and from orders 
ol_df.loc[ol_df['id_order'].isin(temp['id_order'])]

,id,id_order,product_quantity,sku,unit_price,date
117,1119316,299638,1,SYN0127,223.24,2017-01-01 11:56:18
119,1119318,299638,1,WDT0183,151.99,2017-01-01 11:56:50
192,1119477,299706,1,EVU0007,28.49,2017-01-01 13:57:16
198,1119494,299712,1,APP0608,279.99,2017-01-01 14:10:47
510,1120084,300029,1,SYN0127,223.24,2017-01-01 22:00:19
...,...,...,...,...,...,...
241823,1567517,493528,3,WDT0177-A,123.03,2018-01-18 12:40:42
241826,1567521,493528,1,WDT0183,129.99,2018-01-18 12:41:31
263415,1601131,491926,1,REP0088,479.65,2018-02-05 13:17:51
263416,1601133,491926,1,REP0088,14.31,2018-02-05 13:20:30


In [34]:
# deleting from orders 
or_df = or_df.loc[~or_df['order_id'].isin(temp['id_order'])]
or_df.info()

<class 'pandas.DataFrame'>
Index: 46204 entries, 1 to 226619
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      46204 non-null  int64         
 1   created_date  46204 non-null  datetime64[us]
 2   total_paid    46204 non-null  float64       
 3   state         46204 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 1.8 MB


In [35]:
# deleting from order lines 
ol_df = ol_df.loc[ol_df['id_order'].isin(or_df['order_id'])]
ol_df.info()

<class 'pandas.DataFrame'>
Index: 61407 entries, 7 to 293661
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                61407 non-null  int64         
 1   id_order          61407 non-null  int64         
 2   product_quantity  61407 non-null  int64         
 3   sku               61407 non-null  str           
 4   unit_price        61407 non-null  str           
 5   date              61407 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(3), str(2)
memory usage: 3.3 MB


In [36]:
pr_df.info()

<class 'pandas.DataFrame'>
Index: 5985 entries, 0 to 10535
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          5985 non-null   str  
 1   name         5985 non-null   str  
 2   desc         5985 non-null   str  
 3   price        5985 non-null   str  
 4   promo_price  5985 non-null   str  
 5   in_stock     5985 non-null   int64
 6   type         5985 non-null   str  
dtypes: int64(1), str(6)
memory usage: 374.1 KB


In [37]:
# another check on Products df to see which products were also removed from orderlines
never = pr_df.loc[~pr_df['sku'].isin(ol_df['sku'])]
never

,sku,name,desc,price,promo_price,in_stock,type
1025,GRT0338,Griffin Trainer Armband iPhone 6 / 6S Black,Bracelet iPhone 6 / 6S sports waterproof.,19.99,149.895,0,5405
2839,PAC0932,"Apple iMac 27 ""Core i5 3.2GHz Retina 5K | 16GB...",IMac desktop computer 27 inch 5K Retina 16GB 2...,2609,23.469.898,0,1282
2960,APP1241,"Apple iMac 21.5 ""Core i5 2.8GHz | 8GB | 1TB Fu...",PC 215 inch iMac 2.8GHz 8GB RAM 1TB Fusion (MK...,1649,15.735.844,0,1282
4089,MYF0012,MyFox Intellitag® intelligent sensors for Home...,Kit of 5 antitheft security system sensors My ...,199.99,1.999.888,0,11905404
5254,SYN0148,Synology DS416slim NAS server Mac and PC,8TB NAS server storage core 1.0 GHz dual-frequ...,313.99,3.049.901,0,12175397
5891,APP1840,"Apple MacBook Pro 13 ""with Touch Bar 33GHz Cor...",New MacBook Pro 13 inch Touch Bar 33 GHz Core ...,2359,22.475.847,0,2158
6548,PAC1933,"Second hand - Apple iMac 215 ""Core i5 Quad-Cor...",Computer Refurbished iMac Core i5 215 inches 2...,1199,7.289.951,0,"5,72E+15"
7090,NTE0048-A,Open - NewerTech NuGuard KX iPad Air Case Black,IPad case ultra resistant to extreme condition...,79.99,177.425,0,1298
7246,PAC1938,"Second hand - Apple iMac 215 ""Core i5 Quad-Cor...",Computer Refurbished iMac Core i5 215 inches 2...,1199,7.999.894,0,"5,72E+15"
7483,PAC2054,"Second hand - Apple iMac 27 ""Core i5 Quad-Core...",Computer Refurbished iMac 27 inch Core i5 Quad...,1699,1059,0,"5,72E+15"


In [38]:
pr_df = pr_df.loc[~pr_df['sku'].isin(never['sku'])]

In [39]:
ol_df['sku'].nunique()

5968

In [40]:
pr_df['sku'].nunique()

5968

### Fixing Datatypes : 

Some notes:

Our source of truth is `unit_price` in the Order Lines DataFrame because it is the real transaction price used when selling.

Therefore, the trusted source hierarchy is:

1. `unit_price`
2. `promo_price`
3. `price`

`promo_price` is a product-level field, so it should be treated carefully in the analysis. It can help with data quality checks, but the actual discount should mainly be based on `price` and `unit_price`.

In [41]:
ol_df['unit_price'].isna().sum()

np.int64(0)

In [42]:
# Let's start with unit prices.
mask = ol_df['unit_price'].str.count(r'\.') > 1
bad_rows = ol_df.loc[mask].copy()
bad_rows.sort_values('sku')

,id,id_order,product_quantity,sku,unit_price,date
123738,1364981,406149,1,AP20028,1.005.59,2017-09-29 09:29:22
123512,1364588,405962,1,AP20079,1.105.59,2017-09-28 18:16:32
103619,1316450,388716,1,AP20079,1.105.59,2017-08-11 14:20:45
73367,1262304,361840,1,AP20079,1.159.00,2017-06-06 11:11:03
71324,1258847,359758,1,AP20079,1.159.00,2017-05-31 13:30:40
...,...,...,...,...,...,...
65866,1248898,356325,1,WAC0239,1.279.00,2017-05-16 15:51:40
57675,1234704,349374,1,WAC0239,1.589.99,2017-04-24 21:34:07
113887,1347032,397428,1,WAC0243,1.525.59,2017-09-07 13:50:41
79613,1273183,368114,1,WAC0243,1.519.00,2017-06-22 12:33:58


In [43]:
# Correct the malformed unit_price values.
bad_rows.loc[:, "unit_price_clean"] = bad_rows["unit_price"].apply(clean_unit_prices)
bad_rows.sort_values('sku')

,id,id_order,product_quantity,sku,unit_price,date,unit_price_clean
123738,1364981,406149,1,AP20028,1.005.59,2017-09-29 09:29:22,1005.59
123512,1364588,405962,1,AP20079,1.105.59,2017-09-28 18:16:32,1105.59
103619,1316450,388716,1,AP20079,1.105.59,2017-08-11 14:20:45,1105.59
73367,1262304,361840,1,AP20079,1.159.00,2017-06-06 11:11:03,1159.0
71324,1258847,359758,1,AP20079,1.159.00,2017-05-31 13:30:40,1159.0
...,...,...,...,...,...,...,...
65866,1248898,356325,1,WAC0239,1.279.00,2017-05-16 15:51:40,1279.0
57675,1234704,349374,1,WAC0239,1.589.99,2017-04-24 21:34:07,1589.99
113887,1347032,397428,1,WAC0243,1.525.59,2017-09-07 13:50:41,1525.59
79613,1273183,368114,1,WAC0243,1.519.00,2017-06-22 12:33:58,1519.0


In [44]:
# Replace the malformed unit_price values in the main orderlines DataFrame.
price_map = bad_rows.set_index('id')['unit_price_clean']

mask = ol_df['id'].isin(bad_rows['id'])

ol_df.loc[mask, 'unit_price'] = ol_df.loc[mask, 'id'].map(price_map)

In [45]:
ol_df.loc[ol_df['unit_price'].str.count(r"\.") > 1]

,id,id_order,product_quantity,sku,unit_price,date


In [46]:
ol_df['unit_price'] = pd.to_numeric(ol_df['unit_price'])

In [47]:
ol_df['unit_price'].isna().sum()

np.int64(0)

In [48]:
ol_df.info()

<class 'pandas.DataFrame'>
Index: 61407 entries, 7 to 293661
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                61407 non-null  int64         
 1   id_order          61407 non-null  int64         
 2   product_quantity  61407 non-null  int64         
 3   sku               61407 non-null  str           
 4   unit_price        61407 non-null  float64       
 5   date              61407 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(3), str(1)
memory usage: 3.3 MB


#### Promo_price

In [49]:
# Now let's take care of promo_price.
bad_promo_prices = pr_df.loc[pr_df['promo_price'].str.count(r'\.') > 1].copy()
bad_promo_prices

,sku,name,desc,price,promo_price,in_stock,type
50,APP0367,Apple Mini DisplayPort to DVI Adapter Mac dual...,Adapter Mini Display Port to DVI dual channel ...,119,1.119.976,0,1325
67,MAK0007,Maclocks theft case iPad 2 3 and 4 transparent...,Case antitheft iPad 2 3 and 4 polycarbonate ro...,120,1.079.961,0,12635403
116,PAC1240,"Apple MacBook Pro 133 ""i5 25GHz | RAM 16GB | 5...",Apple MacBook Pro 133 inches (MD101Y / A) with...,1899,15.755.945,0,1282
120,APP0490,Apple Airport Express,Airport Express base station 802.11a / b / g / n.,109,1.039.995,0,1334
139,WDT0141,"WD Red 3TB 35 ""Mac PC hard drive and NAS",Western Digital hard drive designed for NAS 3T...,129,1.019.933,1,12655397
...,...,...,...,...,...,...,...
10453,AP20390,Like new - Apple iPad Wi-Fi + Cellular 128GB S...,iPad Wi-Fi + Cellular Refurbished 128GB in Silver,662.81,5.528.115,0,12141714
10464,DLL0050-A,"Open - Dell P2418D 24 ""IPS qHD DP HDMI",QHD Monitor with HDMI DisplayPort ports 4 USB ...,310.99,2.293.778,0,1296
10478,WAC0258,Education - Wacom Intuos Graphics Tablet M Blu...,Medium-edge graphics tablet with Bluetooth int...,199,1.649.896,1,1405
10485,NKI0014-A,Open - Nokia HR 40 mm Steel Smartwatch Smart C...,Nokia smart watch 40 mm Steel HR with heart ra...,199.95,1.336.732,0,11905404


In [50]:
# The first idea was to use the latest unit_price, but this may mix transaction-level and product-level information.

In [51]:
# # let's try to get the unit_price from order lines DataFrame
# # Get the latest unit_price for each sku in sold_products (based on max date)
# filtered = ol_df[ol_df['sku'].isin(bad_promo_prices['sku'])]
# latest_prices = filtered.loc[filtered.groupby('sku')['date'].idxmax()]
# latest_prices

In [52]:
# # we can replace promo_price with the latest unit_price (based on max date)
# correct_unit_price = latest_prices.loc[~(latest_prices['unit_price'].str.count(r'\.') > 1)]

In [53]:
# correct_unit_price.info()

In [54]:
# price_map = correct_unit_price.set_index('sku')['unit_price']

# mask = pr_df['sku'].isin(correct_unit_price['sku'])

# pr_df.loc[mask, 'promo_price'] = (
#     pr_df.loc[mask, 'sku'].map(price_map)
# )

In [50]:
# Try to fix the promo_prices using clean_prices_v2.
bad_promo_prices.loc[:, "promo_price_clean"] = bad_promo_prices["promo_price"].apply(clean_prices_v2)
bad_promo_prices

,sku,name,desc,price,promo_price,in_stock,type,promo_price_clean
50,APP0367,Apple Mini DisplayPort to DVI Adapter Mac dual...,Adapter Mini Display Port to DVI dual channel ...,119,1.119.976,0,1325,111.9976
67,MAK0007,Maclocks theft case iPad 2 3 and 4 transparent...,Case antitheft iPad 2 3 and 4 polycarbonate ro...,120,1.079.961,0,12635403,107.9961
116,PAC1240,"Apple MacBook Pro 133 ""i5 25GHz | RAM 16GB | 5...",Apple MacBook Pro 133 inches (MD101Y / A) with...,1899,15.755.945,0,1282,1575.5945
120,APP0490,Apple Airport Express,Airport Express base station 802.11a / b / g / n.,109,1.039.995,0,1334,103.99949999999998
139,WDT0141,"WD Red 3TB 35 ""Mac PC hard drive and NAS",Western Digital hard drive designed for NAS 3T...,129,1.019.933,1,12655397,101.9933
...,...,...,...,...,...,...,...,...
10453,AP20390,Like new - Apple iPad Wi-Fi + Cellular 128GB S...,iPad Wi-Fi + Cellular Refurbished 128GB in Silver,662.81,5.528.115,0,12141714,552.8115
10464,DLL0050-A,"Open - Dell P2418D 24 ""IPS qHD DP HDMI",QHD Monitor with HDMI DisplayPort ports 4 USB ...,310.99,2.293.778,0,1296,229.37779999999998
10478,WAC0258,Education - Wacom Intuos Graphics Tablet M Blu...,Medium-edge graphics tablet with Bluetooth int...,199,1.649.896,1,1405,164.9896
10485,NKI0014-A,Open - Nokia HR 40 mm Steel Smartwatch Smart C...,Nokia smart watch 40 mm Steel HR with heart ra...,199.95,1.336.732,0,11905404,133.6732


In [51]:
# now replace
promo_price_map = bad_promo_prices.set_index('sku')['promo_price_clean']

mask = pr_df['sku'].isin(bad_promo_prices['sku'])

pr_df.loc[mask, 'promo_price'] = (
    pr_df.loc[mask, 'sku'].map(promo_price_map)
)

In [52]:
pr_df.loc[pr_df['promo_price'].str.count(r'\.') > 1]

,sku,name,desc,price,promo_price,in_stock,type


In [53]:
pr_df['promo_price'].isna().sum()

np.int64(0)

#### Price

In [59]:
# now let us fix datatypes
# pd.to_numeric(pr_df['price'])

In [54]:
# Let's see how many products have malformed price values.
bad_prices = pr_df.loc[pr_df['price'].str.count(r'\.') > 1].copy()
bad_prices

,sku,name,desc,price,promo_price,in_stock,type
505,CRU0015-2,Crucial memory Mac 16GB (2x8GB) SO-DIMM DDR3 1...,RAM 16GB (2x8GB) 135V MacBook Pro iMac (2012/2...,1.639.792,162.9894,1,1364
747,REP0188,Full Screen Repair iPad Mini 2,Repair service including parts and labor for i...,2.099.895,209.9895,0,"1,44E+11"
1190,APP0869,Apple iPad 2 Wi-Fi Air 128GB Space Gray,New iPad Air 2 Wi-Fi 128GB (MGTX2TY / A).,5.428.121,542.8121,0,42931714
1192,APP0870,Apple iPad 2 Wi-Fi Air 128GB Silver,New iPad Air 2 Wi-Fi 128GB (MGTY2TY / A).,5.428.121,542.8121,0,42931714
1194,APP0871,Apple iPad 2 Wi-Fi Air 128GB Dorado,New iPad Air 2 Wi-Fi 128GB (MH1J2TY / A).,5.428.121,542.8121,0,42931714
...,...,...,...,...,...,...,...
9779,WDT0400,WD My Cloud Home 3TB USB 3.0,1-bay NAS server for Mac and PC,2.049.946,204.9946,0,11935397
10086,TRA0044-A,Open - Transcend JetDrive PCIe SSD 820 M13-M15...,Kit 960GB SSD expansion refitted for Macbook P...,7.109.004,597.7884,0,12215397
10127,APP2155-A,Open - Smart Keyboard Apple Keyboard Folio iPa...,Cover with keyboard shortcuts and Spanish (dir...,1.790.001,173.3485,0,12635403
10434,UBI0007,Ubiquiti Amplifi Wi-Fi Mesh Router + 2 Mesh Ac...,Wi-Fi high-density intelligent Mesh technology,35.998.952,359.9895,0,1334


In [55]:
# Correct the malformed price values using clean_prices_v2.
bad_prices.loc[:, "price_clean"] = bad_prices["price"].apply(clean_prices_v2)
bad_prices

,sku,name,desc,price,promo_price,in_stock,type,price_clean
505,CRU0015-2,Crucial memory Mac 16GB (2x8GB) SO-DIMM DDR3 1...,RAM 16GB (2x8GB) 135V MacBook Pro iMac (2012/2...,1.639.792,162.9894,1,1364,163.9792
747,REP0188,Full Screen Repair iPad Mini 2,Repair service including parts and labor for i...,2.099.895,209.9895,0,"1,44E+11",209.9895
1190,APP0869,Apple iPad 2 Wi-Fi Air 128GB Space Gray,New iPad Air 2 Wi-Fi 128GB (MGTX2TY / A).,5.428.121,542.8121,0,42931714,542.8121
1192,APP0870,Apple iPad 2 Wi-Fi Air 128GB Silver,New iPad Air 2 Wi-Fi 128GB (MGTY2TY / A).,5.428.121,542.8121,0,42931714,542.8121
1194,APP0871,Apple iPad 2 Wi-Fi Air 128GB Dorado,New iPad Air 2 Wi-Fi 128GB (MH1J2TY / A).,5.428.121,542.8121,0,42931714,542.8121
...,...,...,...,...,...,...,...,...
9779,WDT0400,WD My Cloud Home 3TB USB 3.0,1-bay NAS server for Mac and PC,2.049.946,204.9946,0,11935397,204.9946
10086,TRA0044-A,Open - Transcend JetDrive PCIe SSD 820 M13-M15...,Kit 960GB SSD expansion refitted for Macbook P...,7.109.004,597.7884,0,12215397,710.9004
10127,APP2155-A,Open - Smart Keyboard Apple Keyboard Folio iPa...,Cover with keyboard shortcuts and Spanish (dir...,1.790.001,173.3485,0,12635403,179.0001
10434,UBI0007,Ubiquiti Amplifi Wi-Fi Mesh Router + 2 Mesh Ac...,Wi-Fi high-density intelligent Mesh technology,35.998.952,359.9895,0,1334,3599.8952


In [56]:
price_map = bad_prices.set_index('sku')['price_clean']

mask = pr_df['sku'].isin(bad_prices['sku'])

pr_df.loc[mask, 'price'] = (
    pr_df.loc[mask, 'sku'].map(price_map)
)

In [57]:
pr_df.loc[pr_df['price'].str.count(r'\.') > 1]

,sku,name,desc,price,promo_price,in_stock,type


In [58]:
# now after we fixed price and promo_price columns let's try and convert them to numeric
pr_df['price'] = pd.to_numeric(pr_df['price'])
pr_df['promo_price'] = pd.to_numeric(pr_df['promo_price'])

In [59]:
# Check the final data types after converting price and promo_price to numeric.
pr_df.info()

<class 'pandas.DataFrame'>
Index: 5968 entries, 0 to 10535
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku          5968 non-null   str    
 1   name         5968 non-null   str    
 2   desc         5968 non-null   str    
 3   price        5968 non-null   float64
 4   promo_price  5968 non-null   float64
 5   in_stock     5968 non-null   int64  
 6   type         5968 non-null   str    
dtypes: float64(2), int64(1), str(4)
memory usage: 373.0 KB


In [60]:
# Quality check: product catalog price vs. actual transaction unit_price.
temp = (
    pr_df.merge(ol_df, on='sku', how='inner')
    .sort_values(['sku', 'date'], ascending=False)
    [['sku', 'price', 'promo_price', 'unit_price', 'date']]
    .drop_duplicates('sku')
    .copy()
)

temp['discount_pct_vs_price'] = (temp["price"] - temp["unit_price"]) / temp["price"] * 100

temp

,sku,price,promo_price,unit_price,date,discount_pct_vs_price
40345,par0072,349.00,209.9895,209.99,2018-01-16 12:05:16,39.830946
9183,ZEP0008,149.99,125.9900,104.12,2017-12-21 00:48:57,30.582039
9171,ZEP0007,149.99,125.9900,125.99,2018-01-02 11:45:29,16.001067
61367,ZAG0042,29.99,199.9040,19.99,2018-03-09 10:43:28,33.344448
61366,ZAG0041,29.99,199.9040,19.99,2018-03-09 10:37:20,33.344448
...,...,...,...,...,...,...
42750,8MO0009,35.00,149.8950,14.70,2018-01-27 19:52:26,58.000000
42743,8MO0008,35.00,199.9040,19.99,2017-08-20 11:27:08,42.885714
42725,8MO0007,35.00,239.8950,19.99,2017-09-05 07:40:35,42.885714
61103,8MO0003-A,35.00,128.5150,12.85,2018-03-08 18:52:29,63.285714


In [61]:
# Products with very high discount percentages may indicate prices that are still scaled incorrectly.
temp.loc[(temp['discount_pct_vs_price'] >= 80) & (temp['price'] / 10 < temp['unit_price'] - 1)]

,sku,price,promo_price,unit_price,date,discount_pct_vs_price
40957,STA0057,58.9900,449.9020,11.25,2018-02-17 18:35:21,80.928971
22414,SAN0138,502.8034,502.8030,53.99,2017-07-04 13:12:09,89.262205
7722,PUR0107,19.9900,39.9060,3.30,2017-04-17 21:16:36,83.491746
56581,PAC2281,1499.0000,255.5945,255.59,2017-11-14 12:47:25,82.949300
6596,OWC0087,55.9900,99.8980,9.99,2017-11-06 14:28:01,82.157528
41281,NTE0039-A,60.9900,78.9860,7.90,2018-02-03 15:18:06,87.047057
29563,LIF0038-A,79.9900,151.6780,12.54,2017-01-29 12:37:20,84.323040
46206,GRT0466,29.9900,44.9030,4.49,2017-11-21 15:09:01,85.028343
46117,GRT0464,29.9900,4.5000,4.50,2018-03-09 23:46:31,84.994998
46250,GRT0463,29.9900,4.9900,4.99,2017-09-02 00:46:42,83.361120


In [62]:
# Catch products where the catalog price still seems much higher than the actual transaction price.
mask = temp['discount_pct_vs_price'] >= 80

still_bad_prices = temp.loc[mask].copy()
still_bad_prices

,sku,price,promo_price,unit_price,date,discount_pct_vs_price
5636,WDT0185,639.8960,705.8410,49.99,2017-01-07 21:01:25,92.187793
44749,WAC0235,199.9041,199.9040,19.99,2018-01-31 15:56:41,90.000205
44655,WAC0234,799.8950,799.8950,79.99,2018-02-05 11:29:17,89.999937
60231,WAC0232-A,349.8960,27.2250,27.23,2018-01-17 17:16:25,92.217688
12353,WAC0142,349.9320,349.9320,34.99,2017-07-21 10:29:20,90.000914
...,...,...,...,...,...,...
24690,APP1498,8328.1154,832.8115,829.00,2017-05-30 10:19:14,90.045767
24415,APP1477,4903.3072,490.3307,459.99,2017-02-13 23:05:00,90.618781
52445,APP1188-A,350.0050,325.3560,32.54,2017-10-25 08:19:14,90.702990
51478,AP20125,35.0000,59.8950,5.99,2017-09-18 13:54:50,82.885714


In [63]:
# Correct the remaining suspicious catalog prices by dividing by 10.
still_bad_prices.loc[:, "price_clean"] = still_bad_prices["price"].apply(clean_still_bad_promo_prices)
still_bad_prices

,sku,price,promo_price,unit_price,date,discount_pct_vs_price,price_clean
5636,WDT0185,639.8960,705.8410,49.99,2017-01-07 21:01:25,92.187793,63.98960
44749,WAC0235,199.9041,199.9040,19.99,2018-01-31 15:56:41,90.000205,19.99041
44655,WAC0234,799.8950,799.8950,79.99,2018-02-05 11:29:17,89.999937,79.98950
60231,WAC0232-A,349.8960,27.2250,27.23,2018-01-17 17:16:25,92.217688,34.98960
12353,WAC0142,349.9320,349.9320,34.99,2017-07-21 10:29:20,90.000914,34.99320
...,...,...,...,...,...,...,...
24690,APP1498,8328.1154,832.8115,829.00,2017-05-30 10:19:14,90.045767,832.81154
24415,APP1477,4903.3072,490.3307,459.99,2017-02-13 23:05:00,90.618781,490.33072
52445,APP1188-A,350.0050,325.3560,32.54,2017-10-25 08:19:14,90.702990,35.00050
51478,AP20125,35.0000,59.8950,5.99,2017-09-18 13:54:50,82.885714,3.50000


In [64]:
price_map = still_bad_prices.set_index('sku')['price_clean']

mask = pr_df['sku'].isin(still_bad_prices['sku'])

pr_df.loc[mask, 'price'] = (
    pr_df.loc[mask, 'sku'].map(price_map)
)

In [65]:
tt = pr_df.loc[(pr_df['promo_price'] > pr_df['price']) & (pr_df['promo_price'] - pr_df['price'] > 1)] 
tt

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,229.997,0,1230
5,APP0073,Apple Composite AV Cable iPhone and iPod white,IPhone and iPod AV Cable Dock to Composite Video.,45.00,420.003,0,1230
...,...,...,...,...,...,...,...
10484,STA0035-A,Open - Startech USB Hub-C or VGA to HDMI 4K US...,Hub with USB-C connection VGA port or HDMI 4K ...,123.99,692.247,0,12585395
10496,ZAG0041,Zagg iPhone Glass Screen Protector + 8 Plus / ...,impact and scratch resistant tempered glass fo...,29.99,199.904,1,13555403
10498,ZAG0042,Zagg Glass Screen Protector + iPhone 8/7 / 6 / 6S,Protector impact and scratch resistant tempere...,29.99,199.904,1,13555403
10514,HOC0027,Hoco Grand Series Metal 38mm Apple Watch Strap...,Stainless steel strap Hoco for Apple Watch 38mm.,65.99,599.906,1,2449


In [66]:
# Calculate the ratio of promo_price to regular price.
ratio = pr_df["promo_price"] / pr_df["price"]

# Catch rows where promo_price still seems scaled incorrectly.
mask = (ratio >= 3.0) & (ratio <= 11.0)

still_bad_promo_prices = pr_df.loc[mask].copy()
still_bad_promo_prices

,sku,name,desc,price,promo_price,in_stock,type
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,589.996,0,13855401
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,569.898,0,1387
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,229.997,0,1230
5,APP0073,Apple Composite AV Cable iPhone and iPod white,IPhone and iPod AV Cable Dock to Composite Video.,45.00,420.003,0,1230
...,...,...,...,...,...,...,...
10484,STA0035-A,Open - Startech USB Hub-C or VGA to HDMI 4K US...,Hub with USB-C connection VGA port or HDMI 4K ...,123.99,692.247,0,12585395
10496,ZAG0041,Zagg iPhone Glass Screen Protector + 8 Plus / ...,impact and scratch resistant tempered glass fo...,29.99,199.904,1,13555403
10498,ZAG0042,Zagg Glass Screen Protector + iPhone 8/7 / 6 / 6S,Protector impact and scratch resistant tempere...,29.99,199.904,1,13555403
10514,HOC0027,Hoco Grand Series Metal 38mm Apple Watch Strap...,Stainless steel strap Hoco for Apple Watch 38mm.,65.99,599.906,1,2449


In [67]:
# Correct the remaining suspicious promo_price values by dividing by 10.
still_bad_promo_prices.loc[:, "promo_price_clean"] = still_bad_promo_prices["promo_price"].apply(clean_still_bad_promo_prices)
still_bad_promo_prices

,sku,name,desc,price,promo_price,in_stock,type,promo_price_clean
0,RAI0007,Silver Rain Design mStand Support,Aluminum support compatible with all MacBook,59.99,499.899,1,8696,49.9899
1,APP0023,Apple Mac Keyboard Keypad Spanish,USB ultrathin keyboard Apple Mac Spanish.,59.00,589.996,0,13855401,58.9996
2,APP0025,Mighty Mouse Apple Mouse for Mac,mouse Apple USB cable.,59.00,569.898,0,1387,56.9898
3,APP0072,Apple Dock to USB Cable iPhone and iPod white,IPhone dock and USB Cable Apple iPod.,25.00,229.997,0,1230,22.9997
5,APP0073,Apple Composite AV Cable iPhone and iPod white,IPhone and iPod AV Cable Dock to Composite Video.,45.00,420.003,0,1230,42.0003
...,...,...,...,...,...,...,...,...
10484,STA0035-A,Open - Startech USB Hub-C or VGA to HDMI 4K US...,Hub with USB-C connection VGA port or HDMI 4K ...,123.99,692.247,0,12585395,69.2247
10496,ZAG0041,Zagg iPhone Glass Screen Protector + 8 Plus / ...,impact and scratch resistant tempered glass fo...,29.99,199.904,1,13555403,19.9904
10498,ZAG0042,Zagg Glass Screen Protector + iPhone 8/7 / 6 / 6S,Protector impact and scratch resistant tempere...,29.99,199.904,1,13555403,19.9904
10514,HOC0027,Hoco Grand Series Metal 38mm Apple Watch Strap...,Stainless steel strap Hoco for Apple Watch 38mm.,65.99,599.906,1,2449,59.9906


In [68]:
promo_price_map = still_bad_promo_prices.set_index('sku')['promo_price_clean']

mask = pr_df['sku'].isin(still_bad_promo_prices['sku'])

pr_df.loc[mask, 'promo_price'] = (
    pr_df.loc[mask, 'sku'].map(promo_price_map)
)

In [69]:
# Remaining cases where promo_price is higher than price.
# These are kept and flagged instead of being deleted, because deleting products would break the link with orderlines.
kk = pr_df.loc[(pr_df['promo_price'] > pr_df['price']) & (pr_df['promo_price'] - pr_df['price'] > 1)].copy()
kk

,sku,name,desc,price,promo_price,in_stock,type
174,NTE0023,NewerTech eSATA Cable Adapter Mac Pro,Add 2 eSATA ports with the adapter cable for M...,30.990,73.895,1,12755395
251,KIN0074,Kingston DataTraveler SE9 8GB USB 2.0 key,8GB USB 2.0 key minimalist design.,4.990,57.802,0,57445397
359,SEV0018,IMac RAM installation service,Installation of RAM in your iMac.,1.999,199.892,0,20642062
362,SEV0021,SSD installation service MacBook Pro Retina,Installing SSD in your MacBook Pro Retina + Da...,4.999,499.851,0,20642062
374,LOG0084,Logitech Ultrathin Keyboard Cover Keyboard Cov...,Ultrathin cover and cover with Bluetooth keybo...,89.990,199.892,0,12575403
...,...,...,...,...,...,...,...
10336,MOP0108,Mophie Powerstation Mini Universal 4000mAh Bat...,external battery capacity 4000mAh output volta...,6.995,79.896,0,1515
10337,MOP0109,Mophie Powerstation Mini Universal 4000mAh Bat...,external battery capacity 4000mAh output volta...,6.995,269.903,0,1515
10349,ZAG0026-A,Open - Zagg Rugged Keyboard Folio iPad Messeng...,Case reconditioned keyboard and adjustable pos...,99.990,254.736,0,12575403
10428,MOP0105,Mophie Powerstation 6000mAh battery Universal ...,external battery capacity 6000mAh output volta...,8.995,359.902,0,1515


In [70]:
# Keep these products, but flag them as suspicious.
# The analysis should mainly rely on price and unit_price, while treating promo_price carefully.
pr_df['promo_price_suspect'] = pr_df['sku'].isin(kk['sku'])
pr_df['promo_price_suspect'].value_counts()

promo_price_suspect
False    5825
True      143
Name: count, dtype: int64

In [71]:
pr_df["price"] = pr_df["price"].round(2)
pr_df["promo_price"] = pr_df["promo_price"].round(2)
ol_df["unit_price"] = ol_df["unit_price"].round(2)

In [72]:
temp = ol_df.merge(pr_df, on='sku', how='inner').sort_values('id_order')[['id_order', 'sku','unit_price', 'price', 'promo_price']]
temp

,id_order,sku,unit_price,price,promo_price
31883,241423,LAC0212,129.16,139.99,114.99
48173,242832,PAR0074,10.77,17.99,10.99
8705,243330,OWC0074,77.99,99.99,99.99
16233,245275,TAD0007,149.00,179.00,149.00
4497,245595,PAC1561,52.99,103.95,59.58
...,...,...,...,...,...
61399,527042,APP0927,13.99,35.00,13.99
61403,527070,APP0698,9.99,25.00,9.99
61404,527074,APP0698,9.99,25.00,9.99
61405,527096,APP0698,9.99,25.00,9.99


In [73]:
br_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 187 entries, 0 to 186
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   short   187 non-null    str  
 1   long    187 non-null    str  
dtypes: str(2)
memory usage: 3.1 KB


In [74]:
br_df.duplicated().sum()

np.int64(0)

In [75]:
br_df['short'].nunique()

187

## Final quality checks before export

In [76]:
final_quality_checks = {
    "orders_without_orderlines": (~or_df["order_id"].isin(ol_df["id_order"])).sum(),
    "orderlines_without_orders": (~ol_df["id_order"].isin(or_df["order_id"])).sum(),
    "orderlines_with_unknown_products": (~ol_df["sku"].isin(pr_df["sku"])).sum(),
    "products_not_sold": (~pr_df["sku"].isin(ol_df["sku"])).sum(),
    "missing_values_orders": or_df.isna().sum().sum(),
    "missing_values_orderlines": ol_df.isna().sum().sum(),
    "missing_values_products": pr_df.isna().sum().sum(),
    "suspect_promo_price_products": pr_df["promo_price_suspect"].sum()
}

pd.Series(final_quality_checks, name="count")

orders_without_orderlines             0
orderlines_without_orders             0
orderlines_with_unknown_products      0
products_not_sold                     0
missing_values_orders                 0
missing_values_orderlines             0
missing_values_products               0
suspect_promo_price_products        143
Name: count, dtype: int64

In [77]:
# Export the cleaned DataFrames
ol_df.to_csv('../data/cleaned/order_lines_new.csv', index= False, encoding='utf-8', na_rep='N/A')
or_df.to_csv('../data/cleaned/orders_new.csv', index= False, encoding='utf-8', na_rep='N/A')
pr_df.to_csv('../data/cleaned/products_new.csv', index= False, encoding='utf-8', na_rep='N/A')
br_df.to_csv('../data/cleaned/brands_new.csv', index= False, encoding='utf-8', na_rep='N/A')